# 🔍 Isolation Forest — Anomaly Detection
**Dataset**: Transaction records (500 rows, 5 features)

This notebook walks through the full pipeline:
1. Data loading & EDA
2. Preprocessing
3. Isolation Forest training & hyperparameter discussion
4. Visualisations
5. Evaluation against ground-truth labels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✅')

## 1. Load & Explore the Dataset

In [ ]:
df = pd.read_csv('../data/transactions.csv')
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
print('Label distribution:')
print(df['label'].value_counts())
print(f'\nMissing values: {df.isnull().sum().sum()}')
df.describe().round(2)

In [ ]:
FEATURE_COLS = ['amount', 'hour', 'duration_sec', 'num_items', 'customer_age']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    for label, color in [('normal', '#34d399'), ('anomaly', '#f87171')]:
        subset = df[df['label'] == label][col]
        axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=label, edgecolor='none')
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('')

axes[-1].set_visible(False)
fig.suptitle('Feature Distributions: Normal vs Anomaly', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7, 5))
corr = df[FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Preprocessing
Isolation Forest is scale-sensitive when combined with PCA, so we apply `StandardScaler`.

In [ ]:
X = df[FEATURE_COLS].copy()
y_true = df['label'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaled shape:', X_scaled.shape)

## 3. Isolation Forest — Hyperparameters Explained

| Parameter | Default | Effect |
|---|---|---|
| `n_estimators` | 100 | More trees → more stable scores, slower training |
| `contamination` | 'auto' | Expected fraction of anomalies; sets decision threshold |
| `max_samples` | 'auto' (256) | Samples per tree; smaller = faster, more randomness |
| `max_features` | 1.0 | Features per tree; lower = more random projections |
| `random_state` | None | Seed for reproducibility |

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────
N_ESTIMATORS   = 100
CONTAMINATION  = 0.10   # ~10 % anomalies expected
MAX_SAMPLES    = 1.0    # use all samples per tree
MAX_FEATURES   = 1.0    # use all features per tree
RANDOM_STATE   = 42
# ────────────────────────────────────────────────────────────────

iforest = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    max_samples=MAX_SAMPLES,
    max_features=MAX_FEATURES,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iforest.fit(X_scaled)
print('Model trained ✅')

In [ ]:
raw_preds  = iforest.predict(X_scaled)          # 1 = normal, -1 = anomaly
raw_scores = iforest.decision_function(X_scaled) # higher = more normal

df['prediction']   = np.where(raw_preds == -1, 'anomaly', 'normal')
df['anomaly_score'] = -raw_scores   # flip: higher = more anomalous

print('Prediction counts:')
print(df['prediction'].value_counts())

## 4. Visualisations

In [ ]:
# Anomaly score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, color in [('normal', '#34d399'), ('anomaly', '#f87171')]:
    subset = df[df['prediction'] == label]['anomaly_score']
    axes[0].hist(subset, bins=30, alpha=0.7, color=color, label=label, edgecolor='none')
axes[0].set_title('Anomaly Score Distribution', fontweight='bold')
axes[0].set_xlabel('Anomaly Score (higher = more anomalous)')
axes[0].legend()

# Amount vs Duration
colors = df['prediction'].map({'normal': '#34d399', 'anomaly': '#f87171'})
axes[1].scatter(df['amount'], df['duration_sec'], c=colors, alpha=0.6, s=20, edgecolors='none')
axes[1].set_xlabel('Amount')
axes[1].set_ylabel('Duration (sec)')
axes[1].set_title('Amount vs Duration — coloured by prediction', fontweight='bold')
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color='#34d399', label='normal'), Patch(color='#f87171', label='anomaly')])

plt.tight_layout()
plt.show()

In [ ]:
# PCA 2D projection
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
print(f'Explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%')

plt.figure(figsize=(9, 6))
colors = df['prediction'].map({'normal': '#34d399', 'anomaly': '#f87171'})
plt.scatter(coords[:, 0], coords[:, 1], c=colors, s=18, alpha=0.7, edgecolors='none')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA 2D Projection — Isolation Forest Predictions', fontweight='bold')
from matplotlib.patches import Patch
plt.legend(handles=[Patch(color='#34d399', label='normal'), Patch(color='#f87171', label='anomaly')])
plt.tight_layout()
plt.show()

In [ ]:
# Box plots per feature
fig, axes = plt.subplots(1, len(FEATURE_COLS), figsize=(16, 4))
for ax, col in zip(axes, FEATURE_COLS):
    data = [df[df['prediction'] == 'normal'][col], df[df['prediction'] == 'anomaly'][col]]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                    boxprops=dict(linewidth=1.2),
                    medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], ['#34d399', '#f87171']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Normal', 'Anomaly'], fontsize=8)
    ax.set_title(col, fontsize=10, fontweight='bold')

fig.suptitle('Feature Distributions by Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Evaluation vs Ground-Truth Labels

In [ ]:
print('Classification Report:')
print(classification_report(df['label'], df['prediction']))

cm = confusion_matrix(df['label'], df['prediction'], labels=['normal', 'anomaly'])
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['normal', 'anomaly'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Effect of contamination parameter on # detected anomalies
contam_values = np.arange(0.02, 0.32, 0.02)
detected = []
for c in contam_values:
    m = IsolationForest(n_estimators=100, contamination=c, random_state=42, n_jobs=-1)
    p = m.fit_predict(X_scaled)
    detected.append((p == -1).sum())

plt.figure(figsize=(9, 4))
plt.plot(contam_values, detected, marker='o', color='#38bdf8', linewidth=2, markersize=7)
plt.axvline(0.10, color='#f87171', linestyle='--', label='chosen (0.10)')
plt.xlabel('contamination')
plt.ylabel('# Anomalies Detected')
plt.title('Effect of contamination Hyperparameter', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print('Top 10 Most Anomalous Transactions:')
df.sort_values('anomaly_score', ascending=False)[[
    'transaction_id', 'amount', 'hour', 'duration_sec',
    'num_items', 'customer_age', 'anomaly_score', 'label'
]].head(10)